In [1]:
!pip install ijson==3.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 2.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import ijson
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from glob import glob

# unless pre preprocessing changes, no need to run above

In [4]:
imu_sensor_locations = ['Pelvis', 'L5', 'L3', 'T12', 'T8', 'Neck', 'Head', 'RightShoulder', 'RightUpperArm', 'RightForeArm', 'RightHand', 'LeftShoulder', 'LeftUpperArm', 'LeftForeArm', 'LeftHand', 'RightUpperLeg', 'RightLowerLeg',
                    'RightFoot', 'RightToe', 'LeftUpperLeg', 'LeftLowerLeg', 'LeftFoot', 'LeftToe']


participant_num = [1, 2, 3, 4, 5, 7, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25]

In [5]:
import os
import pandas as pd
import logging
import glob

# Set up logging
logging.basicConfig(level=logging.INFO)

def merge_csv_files_in_folder(folder_path, imu_sensor_locations):
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

    filtered_files = [file for file in csv_files if any(sensor_location in file for sensor_location in imu_sensor_locations)]

    if filtered_files:
        try:
            df_list = [pd.read_csv(file, low_memory=False) for file in filtered_files]
            merged_df = pd.concat(df_list, ignore_index=True)
            return merged_df
        except FileNotFoundError as e:
            logging.error(f"Error: File not found: {e}")
        except pd.errors.ParserError as e:
            logging.error(f"Error: Parsing error: {e}")
    else:
        logging.warning(f"No relevant CSV files found in {folder_path}")
        return None

In [6]:
# Load and process dataframes
folder_path = "/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/processed data"

insole_df = merge_csv_files_in_folder(folder_path, imu_sensor_locations)


In [7]:
insole_df.sort_values(by=['participant_id', 'time', 'sensor_location'])

,time,participant_id,sensor_location,task,surface,insoles_RightFoot_is_step,insoles_LeftFoot_is_step,insoles_RightFoot_is_lifted,insoles_LeftFoot_is_lifted,orientation__q1,...,insoles__time_to_step,insoles__time_to_lift,xsens_footContacts__Heel,xsens_footContacts__Toe,jointAngle_jS1_x,jointAngle_jS1_y,jointAngle_jS1_z,jointAngleXZY_jS1_x,jointAngleXZY_jS1_y,jointAngleXZY_jS1_z
7277561,0,1,Head,C,walk,True,True,False,False,0.621172,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7253275,0,1,L3,C,walk,True,True,False,False,0.714827,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7180417,0,1,L5,C,walk,True,True,False,False,0.725203,...,NaN,NaN,NaN,NaN,0.785568,-3.825569,1.772979,0.785944,-3.849884,1.772812
7520421,0,1,LeftFoot,C,walk,True,True,False,False,0.650226,...,0.0,133.0,False,False,NaN,NaN,NaN,NaN,NaN,NaN
7617565,0,1,LeftForeArm,C,walk,True,True,False,False,0.528093,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6458380,399517,25,RightToe,C,walk,False,False,False,False,0.866184,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6196176,399517,25,RightUpperArm,C,walk,False,False,False,False,-0.685265,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6410436,399517,25,RightUpperLeg,C,walk,False,False,False,False,-0.856654,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6031348,399517,25,T12,C,walk,False,False,False,False,0.893996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Function to pair heel strikes with the next toe-off event
def pair_heel_strikes_toe_offs(heel_strikes, toe_offs):
    pairs = []
    toe_idx = 0
    # Loop over each heel strike
    for heel in heel_strikes:
        # Find the next toe-off that occurs after the heel strike
        while toe_idx < len(toe_offs) and toe_offs[toe_idx] < heel:
            toe_idx += 1
        if toe_idx < len(toe_offs):
            # Pair found
            pairs.append((heel, toe_offs[toe_idx]))
            toe_idx += 1  # Move to the next toe-off for the next pair
    return pairs


In [9]:
# Initialize or clear gait cycle columns for each new run
insole_df['right_gait_cycle'] = np.nan
insole_df['left_gait_cycle'] = np.nan

for p in insole_df['participant_id'].unique():

    current_foot_sensors_df = insole_df.loc[insole_df['participant_id'] == p]

    # Define heel strikes and toe-offs for the right foot
    right_foot_heel_strikes = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_RightFoot_is_step'] == True) &
        (current_foot_sensors_df['insoles_RightFoot_is_lifted'] == False)]
    right_foot_toe_off = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_RightFoot_is_step'] == False) &
        (current_foot_sensors_df['insoles_RightFoot_is_lifted'] == True)]

    # Define heel strikes and toe-offs for the left foot
    left_foot_heel_strikes = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_LeftFoot_is_step'] == True) &
        (current_foot_sensors_df['insoles_LeftFoot_is_lifted'] == False)]
    left_foot_toe_off = current_foot_sensors_df[
        (current_foot_sensors_df['insoles_LeftFoot_is_step'] == False) &
        (current_foot_sensors_df['insoles_LeftFoot_is_lifted'] == True)]

    # Convert heel strikes and toe-offs to lists
    heel_strike_R = list(right_foot_heel_strikes['time'])
    toe_off_R = list(right_foot_toe_off['time'])

    heel_strike_L = list(left_foot_heel_strikes['time'])
    toe_off_L = list(left_foot_toe_off['time'])

    # Generate pairs of steps for right and left foot
    right_foot_steps = pair_heel_strikes_toe_offs(heel_strike_R, toe_off_R)
    left_foot_steps = pair_heel_strikes_toe_offs(heel_strike_L, toe_off_L)

    # Enumerate and assign step cycles to the DataFrame
    for i, (start, end) in enumerate(right_foot_steps):
        insole_df.loc[
            (insole_df['participant_id'] == p) &
            (insole_df['time'] >= start) &
            (insole_df['time'] <= end), 'right_step_count'] = i + 1

    for i, (start, end) in enumerate(left_foot_steps):
        insole_df.loc[
            (insole_df['participant_id'] == p) &
            (insole_df['time'] >= start) &
            (insole_df['time'] <= end), 'left_step_count'] = i + 1

    print(f"Processed participant {p}")


Processed participant 13
Processed participant 5
Processed participant 3
Processed participant 19
Processed participant 17
Processed participant 10
Processed participant 4
Processed participant 2
Processed participant 18
Processed participant 12
Processed participant 14
Processed participant 24
Processed participant 25
Processed participant 15
Processed participant 1
Processed participant 7
Processed participant 23
Processed participant 8
Processed participant 22
Processed participant 16


In [10]:
insole_df['right_gait_cycle'] = insole_df['right_step_count'].fillna(method='ffill')
insole_df['left_gait_cycle'] = insole_df['left_step_count'].fillna(method='ffill')


<ipython-input-10-7787957a4271>:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  insole_df['right_gait_cycle'] = insole_df['right_step_count'].fillna(method='ffill')
<ipython-input-10-7787957a4271>:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  insole_df['left_gait_cycle'] = insole_df['left_step_count'].fillna(method='ffill')


In [11]:
insole_df['participant_id'].unique()

array([13,  5,  3, 19, 17, 10,  4,  2, 18, 12, 14, 24, 25, 15,  1,  7, 23,
        8, 22, 16])

In [12]:
insole_df.to_csv('/content/drive/MyDrive/Next_Step/Honda: multi_modal_gait_database/merged/all_imu_sensor_df.csv', index=False)